In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q gdown ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.5 MB/s eta 0:00:0000:01


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [4]:
!gdown 18umLJ4x8uHAW3bB9sQWit_hBm27x2fXi -O taco_yolo.zip
!ls -la taco_yolo.zip

Downloading...
From (original): https://drive.google.com/uc?id=18umLJ4x8uHAW3bB9sQWit_hBm27x2fXi
From (redirected): https://drive.google.com/uc?id=18umLJ4x8uHAW3bB9sQWit_hBm27x2fXi&confirm=t&uuid=c7ab945e-222f-4c4e-8bc2-1d1be37ae20d
To: /kaggle/working/taco_yolo.zip
100%|███████████████████████████████████████| 2.62G/2.62G [00:17<00:00, 147MB/s]
-rw-r--r-- 1 root root 2618033595 Aug  3 21:37 taco_yolo.zip


In [5]:
!unzip -o -q taco_yolo.zip -d /kaggle/working/
!find /kaggle/working -iname "data.yaml"

/kaggle/working/taco_yolo/data.yaml


In [6]:
import yaml

data_yaml_path = "/kaggle/working/taco_yolo/data.yaml"
with open(data_yaml_path, "r") as f:
    cfg = yaml.safe_load(f)

cfg["path"] = "/kaggle/working/taco_yolo"

with open(data_yaml_path, "w") as f:
    yaml.dump(cfg, f)

print(cfg)

{'path': '/kaggle/working/taco_yolo', 'train': 'images/train', 'val': 'images/val', 'names': {0: 'Aluminium foil', 1: 'Battery', 2: 'Aluminium blister pack', 3: 'Carded blister pack', 4: 'Other plastic bottle', 5: 'Clear plastic bottle', 6: 'Glass bottle', 7: 'Plastic bottle cap', 8: 'Metal bottle cap', 9: 'Broken glass', 10: 'Food Can', 11: 'Aerosol', 12: 'Drink can', 13: 'Toilet tube', 14: 'Other carton', 15: 'Egg carton', 16: 'Drink carton', 17: 'Corrugated carton', 18: 'Meal carton', 19: 'Pizza box', 20: 'Paper cup', 21: 'Disposable plastic cup', 22: 'Foam cup', 23: 'Glass cup', 24: 'Other plastic cup', 25: 'Food waste', 26: 'Glass jar', 27: 'Plastic lid', 28: 'Metal lid', 29: 'Other plastic', 30: 'Magazine paper', 31: 'Tissues', 32: 'Wrapping paper', 33: 'Normal paper', 34: 'Paper bag', 35: 'Plastified paper bag', 36: 'Plastic film', 37: 'Six pack rings', 38: 'Garbage bag', 39: 'Other plastic wrapper', 40: 'Single-use carrier bag', 41: 'Polypropylene bag', 42: 'Crisp packet', 43: 

In [7]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data="/kaggle/working/taco_yolo/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/kaggle/working/runs_taco",
    name="yolo26n_taco",
)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/taco_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud

In [8]:
model.export(format="onnx", imgsz=640)

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 122 layers, 2,386,536 parameters, 0 gradients, 5.4 GFLOPs

PyTorch: starting from '/kaggle/working/runs_taco/yolo26n_taco/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.2 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 242ms
Prepared 2 packages in 289ms
Installed 2 packages in 13ms
 + onnxruntime==1.28.0
 + onnxslim==0.1.95

requirements: AutoUpdate success ✅ 1.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.95...
ONNX: export success ✅ 3.

'/kaggle/working/runs_taco/yolo26n_taco/weights/best.onnx'

In [9]:
import shutil

save_dir = str(results.save_dir)
print(f"Actual results folder: {save_dir}")

shutil.make_archive("/kaggle/working/taco_results", 'zip', save_dir)
print("Archive ready: /kaggle/working/taco_results.zip")

Actual results folder: /kaggle/working/runs_taco/yolo26n_taco
Archive ready: /kaggle/working/taco_results.zip
